In [13]:
!pip install -q youtube-transcript-api langchain-community langchain-openai faiss-cpu tiktoken python-dotenv

In [12]:
!pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 530.8 kB/s  0:00:03 eta 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 25.3
    Uninstalling pip-25.3:
      Successfully uninstalled pip-25.3


In [14]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

#### Step 1a - Indexing (Document Ingestion)

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

video_id = "b-Pn0yXL9y8" # only the ID, not full URL

try:
    # 1. Instantiate the API client object
    client = YouTubeTranscriptApi() 
    
    # FIXED: Changed 'get_transcript' to 'fetch'
    transcript_list = client.fetch(video_id, languages=['en'])
    
    # 2. Convert the FetchedTranscript object into a raw list of dictionaries
    raw_data = transcript_list.to_raw_data()
    transcript= " ".join(chunk['text'] for chunk in raw_data)
    
    # Print the first two elements to verify it works
    print(raw_data) 
    # print(transcript)
    
except TranscriptsDisabled:
    print("Transcripts are disabled for this video.")


[{'text': 'if you want to change the world start', 'start': 0.77, 'duration': 9.34}, {'text': 'off by making your bed if you make your', 'start': 4.529, 'duration': 7.501}, {'text': 'bed every morning you will have', 'start': 10.11, 'duration': 4.159}, {'text': 'accomplished the first task of the day', 'start': 12.03, 'duration': 4.68}, {'text': 'it will give you a small sense of pride', 'start': 14.269, 'duration': 4.631}, {'text': 'and it will encourage you to do another', 'start': 16.71, 'duration': 6.059}, {'text': 'task and another and another and by the', 'start': 18.9, 'duration': 5.43}, {'text': 'end of the day that one task completed', 'start': 22.769, 'duration': 3.361}, {'text': 'will have turned into mini task', 'start': 24.33, 'duration': 4.08}, {'text': 'completed making your bed will also', 'start': 26.13, 'duration': 4.62}, {'text': 'reinforce the fact that the little', 'start': 28.41, 'duration': 4.68}, {'text': "things in life matter if you can't do", 'start': 30.75, 

In [29]:
transcript

"if you want to change the world start off by making your bed if you make your bed every morning you will have accomplished the first task of the day it will give you a small sense of pride and it will encourage you to do another task and another and another and by the end of the day that one task completed will have turned into mini task completed making your bed will also reinforce the fact that the little things in life matter if you can't do the little things right you'll never be able to do the big things right and if by chance you have a miserable day you will come home to a bed that is made that you made and a made bed gives you encouragement that tomorrow will be better I've been a Navy SEAL for 36 years every morning in SEAL training my instructors who at the time were all Vietnam veterans would show up in my barracks room and the first thing they do is inspect my bed if you did it right the corners would be square the covers would be pulled tight the pillows centered just und

In [81]:
print(transcript.find('United'))

-1


### Step 1b - Indexing (Text Splitting)

In [28]:
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)
chunks = splitter.create_documents([transcript])
len(chunks)

41

In [30]:
chunks[20]

Document(metadata={}, page_content="lives forever you're wrong I saw it happen every day in Iraq and Afghanistan but changing the world")

### Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [31]:
embeddings= OpenAIEmbeddings(model= 'text-embedding-3-small')
vector_stor= FAISS.from_documents(chunks, embeddings)


In [32]:
vector_stor.index_to_docstore_id

{0: '555450c3-4f56-4f27-aa9d-4f5d58ee163a',
 1: '3d83c337-7b1e-4acf-b658-c55a825d5f9c',
 2: 'bd5edc58-05e7-410e-9233-2591c5ad888d',
 3: 'b14fa15d-500e-4bba-936f-b116177f3a03',
 4: '267bc803-e808-4dca-a6a2-52e8d4044fae',
 5: 'b15f1543-711b-4f6c-9ffe-4be633d19065',
 6: 'f45ead65-f570-43e5-88e6-b2eafdd232ae',
 7: '4c9988dc-7cd8-4e48-899c-f574b6048937',
 8: '7f894774-6a6f-4687-9dd1-74f94b158a22',
 9: '6b2503c6-1fd7-4adb-ad99-903a2c109446',
 10: '10a6a5aa-580b-4de3-9729-48f5142c73ee',
 11: '4e7009d6-5977-4c2a-8036-3b0611125e24',
 12: '06d16936-60fd-464e-b049-6b7620676e64',
 13: '1bd2d28c-2493-4ee1-98dc-6a75aa079874',
 14: 'd45e68d4-c030-4d58-838c-193cde14d449',
 15: '829cba7f-a86a-4569-9fe4-72342c49a34b',
 16: '2f43ae43-2783-4542-90ca-67eebdc0d62e',
 17: 'd687fe3c-e3a8-4090-abbf-ca7c73b100ee',
 18: '48eb318b-75f1-4d4d-a8ca-b286a40d0d05',
 19: 'f531969e-98b1-4ec4-8b9a-99487aa6cecd',
 20: '7c214466-fbc4-46fb-b16e-ecb0e5845667',
 21: '93d60594-01c1-41a0-a4e3-d1a83309a3b0',
 22: 'a198c1a1-e893-

In [35]:
vector_stor.get_by_ids(["267bc803-e808-4dca-a6a2-52e8d4044fae"])

[Document(id='267bc803-e808-4dca-a6a2-52e8d4044fae', metadata={}, page_content="will also reinforce the fact that the little things in life matter if you can't do the little")]

### Step 2 - Retrieval

In [37]:
retriever= vector_stor.as_retriever(search_type= "similarity", search_kwargs= {"k":4} )

In [39]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7ba2afdb46b0>, search_kwargs={'k': 4})

In [40]:
retriever.invoke('Speaker was for how many years in navy seal and in which locations he has been during service period?')

[Document(id='7f894774-6a6f-4687-9dd1-74f94b158a22', metadata={}, page_content='every morning in SEAL training my instructors who at the time were all Vietnam veterans would show'),
 Document(id='4c9988dc-7cd8-4e48-899c-f574b6048937', metadata={}, page_content="bed gives you encouragement that tomorrow will be better I've been a Navy SEAL for 36 years every"),
 Document(id='d45e68d4-c030-4d58-838c-193cde14d449', metadata={}, page_content='aspiring to be real warriors tough battle-hardened seals but the wisdom of this simple act has been'),
 Document(id='51773f92-6048-40f2-bd96-4ecf4f252ce5', metadata={}, page_content='and what started here will indeed have changed the world for the better finally a seal training')]

In [41]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [49]:
question          = "Is here discussion about the training of the navy seals in Veitnam? if yes then what was discussed"

retrieved_docs    = retriever.invoke(question)

In [50]:
retrieved_docs

[Document(id='7f894774-6a6f-4687-9dd1-74f94b158a22', metadata={}, page_content='every morning in SEAL training my instructors who at the time were all Vietnam veterans would show'),
 Document(id='51773f92-6048-40f2-bd96-4ecf4f252ce5', metadata={}, page_content='and what started here will indeed have changed the world for the better finally a seal training'),
 Document(id='4c9988dc-7cd8-4e48-899c-f574b6048937', metadata={}, page_content="bed gives you encouragement that tomorrow will be better I've been a Navy SEAL for 36 years every"),
 Document(id='d45e68d4-c030-4d58-838c-193cde14d449', metadata={}, page_content='aspiring to be real warriors tough battle-hardened seals but the wisdom of this simple act has been')]

In [52]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"every morning in SEAL training my instructors who at the time were all Vietnam veterans would show\n\nand what started here will indeed have changed the world for the better finally a seal training\n\nbed gives you encouragement that tomorrow will be better I've been a Navy SEAL for 36 years every\n\naspiring to be real warriors tough battle-hardened seals but the wisdom of this simple act has been"

In [54]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [55]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      every morning in SEAL training my instructors who at the time were all Vietnam veterans would show\n\nand what started here will indeed have changed the world for the better finally a seal training\n\nbed gives you encouragement that tomorrow will be better I've been a Navy SEAL for 36 years every\n\naspiring to be real warriors tough battle-hardened seals but the wisdom of this simple act has been\n      Question: Is here discussion about the training of the navy seals in Veitnam? if yes then what was discussed\n    ")

### Step 4 - Generation

In [56]:
from langchain_openai import ChatOpenAI
model= ChatOpenAI()

answer = model.invoke(final_prompt)
print(answer.content)

Yes, there is discussion about Navy SEAL training in Vietnam. The instructors were Vietnam veterans, and the speaker talks about how the training changed the world for the better.


### Building a Chain

In [57]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableSequence, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [58]:
def format_docs(retrieved_docs):
    context_text= "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text


In [ ]:
parallel_chain = RunnableParallel({
    # Run the retriever using the query string
    "context": lambda x: retriever.invoke(x), 
    
    # Pass the query string straight through to the prompt template
    "question": lambda x: x 
})

In [82]:
parallel_chain.invoke('what is the topic of discussion in this video?')

{'context': [Document(id='48eb318b-75f1-4d4d-a8ca-b286a40d0d05', metadata={}, page_content='those struggles and to move forward changing ourselves and changing the world around us will apply'),
  Document(id='d687fe3c-e3a8-4090-abbf-ca7c73b100ee', metadata={}, page_content='or your social status our struggles in this world are similar and the lessons to overcome those'),
  Document(id='93d60594-01c1-41a0-a4e3-d1a83309a3b0', metadata={}, page_content='the world can happen anywhere and anyone can do it so what starts here can indeed change the world'),
  Document(id='51773f92-6048-40f2-bd96-4ecf4f252ce5', metadata={}, page_content='and what started here will indeed have changed the world for the better finally a seal training')],
 'question': 'what is the topic of discussion in this video?'}

In [83]:
parser = StrOutputParser()
main_chain = parallel_chain | prompt | model | parser

In [84]:
main_chain.invoke('Can you summarize the video')

'The video is about how starting something small or making a change in your life can have a positive impact on the world. It mentions the importance of perseverance and not giving up easily.'

In [85]:
main_chain.invoke("what is the topic of discussion in this videos?")

'The topic of discussion in this video is struggles in the world and lessons on how to overcome them to change ourselves and the world around us.'